In [0]:
%sql


truncate table vinoworld.audit.pipeline_log;
truncate table vinoworld.audit.pipeline_step_log;
truncate table vinoworld.audit.ingestion_log;

truncate table vinoworld.bronze.sales_arancione;
truncate table vinoworld.bronze.sales_celeste;
truncate table vinoworld.bronze.sales_verde;
truncate table vinoworld.bronze.products;

truncate table vinoworld.silver.dim_currency;
truncate table vinoworld.silver.dim_date;
truncate table vinoworld.silver.dim_exchange_rate;
truncate table vinoworld.silver.dim_region;
truncate table vinoworld.silver.dim_store;
truncate table vinoworld.silver.dim_territory ;
truncate table vinoworld.silver.dim_product;
truncate table vinoworld.silver.sales;

truncate table vinoworld.gold.sales_fact;



In [0]:

import sys
sys.path.append("/Workspace/Shared")

import uuid
import time
from datetime import datetime, timezone
from pyspark.sql.functions import current_timestamp, lit

%load_ext autoreload
%autoreload 2
from pipeline_logging import pipeline_log_upsert
# from pipeline_utils import get_notebook_context, capture_exception, move_all_files

import pipeline_utils as Utils


PIPELINE_RUN_ID = str(uuid.uuid4())
PIPELINE_START_TS = datetime.now(timezone.utc)
PIPELINE_NAME = "vinoworld_bronze_load"
PIPELINE_STATUS = "running"
PIPELINE_END_TS = None 
ERROR_MESSAGE = None

try:
    pipeline_log_upsert(spark, PIPELINE_RUN_ID, PIPELINE_NAME, PIPELINE_STATUS, PIPELINE_START_TS)

    # Shared parameters passed to every child notebook

    shared_params = {
        "pipeline_run_id": PIPELINE_RUN_ID, 
    }

  #  notebooks_to_run =[ "brz_02_celeste_sales", "brz_03_verde_sales", "brz_01_arancione_sales", "brz_04_products"]
    BRONZE_NOTEBOOKS = [
    "000-MoveFilesFromArchiveToBronze",
    "bronze/brz_01_arancione_sales",
    "bronze/brz_02_celeste_sales",
    "bronze/brz_03_verde_sales",
    "bronze/brz_04_products",
    ]
    
    SILVER_NOTEBOOKS = [
        "silver/slvr_01_load_dim_fromcsv",
        "silver/slvr_02_load_dim_product",
        "silver/slvr_03_load_dim_region",
        "silver/slvr_04_load_sales",
    ]

    GOLD_NOTEBOOKS = [
        "gold/gold_01_load_sales_fact",
    ]


  #  for nb in notebooks_to_run:
  #      print(f"\n--- Running {nb} ---")
  #      result = dbutils.notebook.run(nb, timeout_seconds=1800, arguments=shared_params)
  #      print(f"[{nb}] returned: {result}")


    PIPELINE = [
        ("BRONZE", BRONZE_NOTEBOOKS),
        ("SILVER", SILVER_NOTEBOOKS),
        ("GOLD", GOLD_NOTEBOOKS),
    ]

    for stage_name, notebooks in PIPELINE:
        errors = Utils.run_stage(dbutils, stage_name, notebooks, shared_params)

        if errors:
            raise RuntimeError(f"Pipeline failed in {len(errors)} notebook(s): {errors}")

    PIPELINE_STATUS = "succeeded"
    PIPELINE_END_TS = datetime.now(timezone.utc)
    pipeline_log_upsert(spark, PIPELINE_RUN_ID, PIPELINE_NAME, PIPELINE_STATUS, PIPELINE_START_TS, PIPELINE_END_TS, ERROR_MESSAGE)


except Exception as e:
    err = Utils.capture_exception(e)
    error_message = (
        f"{err['error_type']}: {err['error_message']}\n\n"
        f"{err['error_traceback']}"
    )

    PIPELINE_STATUS = "failed"
    PIPELINE_END_TS = datetime.now(timezone.utc)
    ERROR_MESSAGE = error_message
    pipeline_log_upsert(spark, PIPELINE_RUN_ID, PIPELINE_NAME, PIPELINE_STATUS, PIPELINE_START_TS, PIPELINE_END_TS, ERROR_MESSAGE)

    print(f"[vinoworld_bronze_load] Pipeline Orchestrator failed..\n{error_message}")
    raise
    


In [0]:
%skip
print(f"\nPipeline run {int(PIPELINE_START_TS.strftime('%Y%m%d%H%M%S%f'))} complete.")

test1 = int(PIPELINE_START_TS.strftime('%Y%m%d%H%M%S%f'))

print(f" timestamp as int = {test1}")

In [0]:
%sql
SELECT 'bronze.products row count = '      || CAST(COUNT(*) AS STRING) FROM vinoworld.bronze.products
UNION ALL
SELECT 'bronze.sales_arancione row count = '      || CAST(COUNT(*) AS STRING) FROM vinoworld.bronze.sales_arancione
UNION ALL
SELECT 'bronze.sales_celeste row count = '      || CAST(COUNT(*) AS STRING) FROM vinoworld.bronze.sales_celeste
UNION ALL
SELECT 'bronze.sales_verde row count = '      || CAST(COUNT(*) AS STRING) FROM vinoworld.bronze.sales_verde
UNION ALL
SELECT 'dim_currency row count = '      || CAST(COUNT(*) AS STRING) FROM vinoworld.silver.dim_currency
UNION ALL
SELECT 'dim_date row count = '          || CAST(COUNT(*) AS STRING) FROM vinoworld.silver.dim_date
UNION ALL
SELECT 'dim_exchange_rate row count = ' || CAST(COUNT(*) AS STRING) FROM vinoworld.silver.dim_exchange_rate
UNION ALL
SELECT 'dim_store row count = '         || CAST(COUNT(*) AS STRING) FROM vinoworld.silver.dim_store
UNION ALL
SELECT 'dim_territory row count = '     || CAST(COUNT(*) AS STRING) FROM vinoworld.silver.dim_territory
UNION ALL
SELECT 'dim_product row count = '      || CAST(COUNT(*) AS STRING) FROM vinoworld.silver.dim_product
UNION ALL
SELECT 'silver.sales row count = '      || CAST(COUNT(*) AS STRING) FROM vinoworld.silver.sales
UNION ALL
SELECT 'gold.sales_fact row count = '      || CAST(COUNT(*) AS STRING) FROM vinoworld.gold.sales_fact
